# Natural-Language Data Preparation Workshop

This notebook turns a plain-language request into visible, repeatable data-preparation steps. Each code cell does one job. The current run profiles the training period, records conservative cleaning decisions, and builds provisional feature matrices. It does not train a model.

## Request in plain language

> Load the complete bookings data, split it using the arrival-date periods in the project contract, inspect data quality and target balance on training data, record any cleaning decision, and prepare a small provisional feature set. Do not inspect proof cancellation outcomes or train a model.

**Period rules:** train before 2016-07-01; choose from 2016-07-01 through 2016-12-31; proof from 2017-01-01 onward.

The proof DataFrame is used only to prepare feature rows and report row/date coverage. This notebook does not read its `is_canceled` values. The splits remain in memory; no partition CSV files are written.

## 1. Load the raw file and supplied contract helpers

The raw table is kept unchanged. We add the arrival date to a separate working copy.

In [1]:
from pathlib import Path
import sys

# Find the repository so this works from the repo folder or notebooks/.
repo_root = None
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "src" / "data.py").exists():
        repo_root = folder
        break

if repo_root is None:
    raise FileNotFoundError("Could not find the project src/ folder.")

source_folder = str(repo_root / "src")
if source_folder not in sys.path:
    sys.path.insert(0, source_folder)

import contract
import data
import numpy as np
import pandas as pd

raw_bookings = data.read_raw()
bookings = raw_bookings.copy()
bookings["arrival_date"] = data.arrival_date(bookings)

print(f"Rows: {len(raw_bookings):,}")
print(f"Source columns: {raw_bookings.shape[1]}")
print(f"Arrival dates: {bookings['arrival_date'].min().date()} to {bookings['arrival_date'].max().date()}")

Rows: 119,390
Source columns: 32
Arrival dates: 2015-07-01 to 2017-08-31


## 2. Split by arrival date

The contract function sets the boundaries. The summary below does not calculate cancellation rates.

In [2]:
train, choose, proof = contract.split_periods(bookings)

def summarize_period(name, period):
    """Return rows, dates, and hotel names without inspecting outcomes."""
    return {
        "period": name,
        "rows": len(period),
        "first_arrival": period["arrival_date"].min().date(),
        "last_arrival": period["arrival_date"].max().date(),
        "hotels": sorted(period["hotel"].dropna().unique()),
    }

period_summary = pd.DataFrame([
    summarize_period("train", train),
    summarize_period("choose", choose),
    summarize_period("proof", proof),
])
print(period_summary.to_string(index=False))

print(f"Rows assigned: {len(train) + len(choose) + len(proof):,} of {len(bookings):,}")

period  rows first_arrival last_arrival                     hotels
 train 49157    2015-07-01   2016-06-30 [City Hotel, Resort Hotel]
choose 29546    2016-07-01   2016-12-31 [City Hotel, Resort Hotel]
 proof 40687    2017-01-01   2017-08-31 [City Hotel, Resort Hotel]
Rows assigned: 119,390 of 119,390


## 3. Natural-language request: profile training data quality

> On the training period only, show exact duplicate rows, columns with missing values, and simple range checks. Do not remove rows or fill values yet.

This keeps proof outcomes out of the EDA. Duplicate rows are not automatically errors because the file has no reservation ID to distinguish a repeated record from two identical bookings.

In [3]:
print(f"Training shape: {train.shape[0]:,} rows x {train.shape[1]} columns")
print(f"Exact repeated rows after first occurrence: {train.duplicated().sum():,}")

missing_counts = train.isna().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)
print("Missing values in training data:")
print(missing_counts.to_string())

range_columns = [
    "adults", "children", "babies",
    "stays_in_weekend_nights", "stays_in_week_nights",
    "adr", "previous_cancellations", "previous_bookings_not_canceled",
]
range_summary = train[range_columns].agg(["min", "max"]).T
range_summary["negative_values"] = train[range_columns].lt(0).sum()
print("Range checks (training period):")
print(range_summary.to_string())

Training shape: 49,157 rows x 33 columns
Exact repeated rows after first occurrence: 16,291
Missing values in training data:
company     46123
agent        6960
country       284
children        4
Range checks (training period):
                                min     max  negative_values
adults                          0.0    55.0                0
children                        0.0    10.0                0
babies                          0.0    10.0                0
stays_in_weekend_nights         0.0    18.0                0
stays_in_week_nights            0.0    42.0                0
adr                             0.0  5400.0                0
previous_cancellations          0.0    26.0                0
previous_bookings_not_canceled  0.0    36.0                0


### 3a. Profile all source columns on train

This compact profile shows type, missingness, distinct values, and the three most common values for every source column. The numeric table adds min, median, 99th percentile, max, and negative-value count. All statistics here use train only.

In [4]:
train_source = train[raw_bookings.columns]
column_rows = []
for column in train_source.columns:
    values = train_source[column]
    common_values = values.value_counts(dropna=False).head(3).to_dict()
    column_rows.append({
        "column": column,
        "type": str(values.dtype),
        "missing": int(values.isna().sum()),
        "distinct": int(values.nunique(dropna=True)),
        "top_values": str(common_values),
    })

column_profile = pd.DataFrame(column_rows)
print(column_profile.to_string(index=False))

numeric_columns = train_source.select_dtypes(include="number").columns
numeric_profile = train_source[numeric_columns].quantile([0, 0.5, 0.99, 1]).T
numeric_profile.columns = ["min", "median", "p99", "max"]
numeric_profile["negative_values"] = train_source[numeric_columns].lt(0).sum()
print("\nNumeric source-column summary:")
print(numeric_profile.to_string())

                        column    type  missing  distinct                                                       top_values
                         hotel     str        0         2                     {'City Hotel': 31623, 'Resort Hotel': 17534}
                   is_canceled   int64        0         2                                             {0: 31572, 1: 17585}
                     lead_time   int64        0       379                                      {0: 3338, 1: 1764, 2: 1021}
             arrival_date_year   int64        0         2                                       {2016: 27161, 2015: 21996}
            arrival_date_month     str        0        12                       {'May': 5478, 'April': 5428, 'June': 5292}
      arrival_date_week_number   int64        0        53                                   {25: 1468, 21: 1452, 18: 1437}
     arrival_date_day_of_month   int64        0        31                                    {17: 2147, 5: 1975, 19: 1852}
       stays_in_

### 3b. Check cancellation patterns by training month

This checks whether the target varies over the training timeline and by hotel. It does not use choose or proof outcomes.

In [5]:
train_monthly = (
    train.groupby([train["arrival_date"].dt.to_period("M"), "hotel"])["is_canceled"]
    .agg(["count", "mean"])
    .rename(columns={"count": "bookings", "mean": "cancellation_rate"})
)
print(train_monthly.to_string(formatters={"cancellation_rate": "{:.1%}".format}))

                           bookings cancellation_rate
arrival_date hotel                                   
2015-07      City Hotel        1398             67.2%
             Resort Hotel      1378             23.2%
2015-08      City Hotel        2480             49.7%
             Resort Hotel      1409             26.0%
2015-09      City Hotel        3529             43.7%
             Resort Hotel      1585             34.8%
2015-10      City Hotel        3386             39.0%
             Resort Hotel      1571             26.2%
2015-11      City Hotel        1235             24.4%
             Resort Hotel      1105             16.7%
2015-12      City Hotel        1654             40.4%
             Resort Hotel      1266             24.1%
2016-01      City Hotel        1364             32.1%
             Resort Hotel       884             13.5%
2016-02      City Hotel        2371             39.2%
             Resort Hotel      1520             26.8%
2016-03      City Hotel     

## 4. Inspect training target and provisional input candidates

The target is examined on train only. The candidate fields below are the learner's Bin 1 choices; they are not yet a verified production feature set.

In [6]:
target_counts = train["is_canceled"].value_counts().sort_index()
print("Training target counts:")
print(target_counts.to_string())
print(f"Training cancellation rate: {train['is_canceled'].mean():.1%}")

hotel_target_summary = train.groupby("hotel")["is_canceled"].agg(["count", "mean"])
hotel_target_summary = hotel_target_summary.rename(columns={"count": "bookings", "mean": "cancellation_rate"})
print("Training target by hotel:")
print(hotel_target_summary.to_string(formatters={"cancellation_rate": "{:.1%}".format}))

candidate_columns = [
    "hotel", "is_repeated_guest",
    "previous_cancellations", "previous_bookings_not_canceled",
]
candidate_profile = train[candidate_columns].agg(["count"]).T
candidate_profile["missing"] = train[candidate_columns].isna().sum()
candidate_profile["distinct_values"] = train[candidate_columns].nunique(dropna=True)
candidate_profile["zero_rows"] = train[candidate_columns].eq(0).sum()
print("Provisional candidate fields:")
print(candidate_profile.to_string())

Training target counts:
is_canceled
0    31572
1    17585
Training cancellation rate: 35.8%
Training target by hotel:
              bookings cancellation_rate
hotel                                   
City Hotel       31623             41.7%
Resort Hotel     17534             25.2%
Provisional candidate fields:
                                count  missing  distinct_values  zero_rows
hotel                           49157        0                2          0
is_repeated_guest               49157        0                2      47529
previous_cancellations          49157        0               14      43348
previous_bookings_not_canceled  49157        0               37      47813


## 5. Decide whether to clean anything

Based on the checks above, this run makes a conservative choice: **do not drop rows or recode raw values**. There are exact duplicate rows, but no reservation identifier proves they are erroneous; the candidate features have no missing values; and unusual but non-negative values are not enough evidence to clip. Missing values in `agent`, `company`, `country`, and `children` are left as recorded because they are not in this provisional feature set.

The raw table remains unchanged. Feature engineering below creates a separate feature table.

In [7]:
candidate_missing = pd.DataFrame({
    "train": train[candidate_columns].isna().sum(),
    "choose": choose[candidate_columns].isna().sum(),
    "proof": proof[candidate_columns].isna().sum(),
})
print("Missing values in provisional inputs by split:")
print(candidate_missing.to_string())

cleaning_decisions = pd.DataFrame([
    {"choice": "Drop rows", "rows_changed": 0, "reason": "No evidenced row-level error or booking ID to confirm duplicate records."},
    {"choice": "Fill missing candidate inputs", "rows_changed": 0, "reason": "The table above shows no missing values in the provisional inputs."},
    {"choice": "Clip unusual numeric values", "rows_changed": 0, "reason": "No negative values were found in the checked training fields; large values alone do not prove an error."},
])
print(cleaning_decisions.to_string(index=False))

Missing values in provisional inputs by split:
                                train  choose  proof
hotel                               0       0      0
is_repeated_guest                   0       0      0
previous_cancellations              0       0      0
previous_bookings_not_canceled      0       0      0
                       choice  rows_changed                                                                                                  reason
                    Drop rows             0                                No evidenced row-level error or booking ID to confirm duplicate records.
Fill missing candidate inputs             0                                      The table above shows no missing values in the provisional inputs.
  Clip unusual numeric values             0 No negative values were found in the checked training fields; large values alone do not prove an error.


## 6. Select a provisional feature set and engineer simple features

> Use the learner-audited Bin 1 fields as the candidate pool. One-hot encode `hotel` using training categories only, and add `log1p` versions of the prior booking counts because the training counts are strongly concentrated at zero. Do not include `is_canceled` as an input.

These features are prepared for later model comparison, not declared production-safe. The timing of historical fields must still be verified against the intended decision timestamp.

In [8]:
hotel_categories = sorted(train["hotel"].dropna().unique())

def make_features(period):
    """Build the same candidate features without reading the target."""
    features = period[candidate_columns].copy()
    features["log_previous_cancellations"] = np.log1p(features["previous_cancellations"])
    features["log_previous_bookings_not_canceled"] = np.log1p(features["previous_bookings_not_canceled"])
    features = features.drop(columns="hotel")

    for hotel_name in hotel_categories:
        column_name = "hotel_" + hotel_name.lower().replace(" ", "_")
        features[column_name] = period["hotel"].eq(hotel_name).astype(int)

    return features

# Prepare features separately for each time period. No target is passed in.
X_train = make_features(train)
X_choose = make_features(choose).reindex(columns=X_train.columns, fill_value=0)
X_proof = make_features(proof).reindex(columns=X_train.columns, fill_value=0)

print("Feature columns:")
print(X_train.columns.tolist())
print(f"Train feature shape: {X_train.shape}")
print(f"Choose feature shape: {X_choose.shape}")
print(f"Proof feature shape: {X_proof.shape}")
print(f"Missing values in train features: {int(X_train.isna().sum().sum())}")
print(f"Missing values in choose features: {int(X_choose.isna().sum().sum())}")
print(f"Missing values in proof features: {int(X_proof.isna().sum().sum())}")

Feature columns:
['is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'log_previous_cancellations', 'log_previous_bookings_not_canceled', 'hotel_city_hotel', 'hotel_resort_hotel']
Train feature shape: (49157, 7)
Choose feature shape: (29546, 7)
Proof feature shape: (40687, 7)
Missing values in train features: 0
Missing values in choose features: 0
Missing values in proof features: 0


## Results and what remains

- The contract split is complete, and all rows are assigned.
- Training EDA and the no-row-removal/no-recoding decision are recorded above. Exact duplicate rows are retained because the available columns cannot confirm they are duplicate reservations.
- The provisional feature matrices are prepared from the learner-audited candidate pool. The target is kept separate.
- No model has been trained. Do not use proof outcomes until the feature set, model, and buffer are frozen.

Next: review these results, confirm whether to keep the provisional feature set, and then compare the planned models using the contract split.